<a href="https://colab.research.google.com/github/IKKNIGHT/MYC-PROTAC/blob/main/MYC_Substrate_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys, torch

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

TORCH_VER  = torch.__version__.split('+')[0]
CUDA_VER   = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
PYG_URL    = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'

# Essential PyG extensions for knn_graph and message passing
pip('torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv', '-f', PYG_URL)
pip('torch-geometric', 'biopython', 'freesasa', 'trimesh', 'pdb2pqr', 'scipy', 'tqdm')


# Mount Drive for checkpoints
from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print('Drive already mounted or unavailable.')

CHECKPOINT_DIR = '/content/drive/MyDrive/myc_gnn/checkpoints'
import os; os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

Mounted at /content/drive
PyTorch 2.10.0+cpu | CUDA: False


This should just download everything

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import os, subprocess, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn  import GCNConv, BatchNorm, global_mean_pool
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn  import knn_graph
from torch_geometric.utils import to_undirected

import freesasa
import trimesh
import trimesh.curvature as tc
from scipy.spatial import cKDTree
from scipy.interpolate import RegularGridInterpolator
from Bio import PDB
from tqdm import tqdm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on:', device)

Running on: cpu


In [ ]:
# ── Cell 3: Download training structures ──────────────────────────────────────
import urllib.request

# (PDB_ID, chain_A, chain_B)
# chain_A = the chain your "interface positive" patches come from
# chain_B = the partner chain used to define the interface
TRAINING_STRUCTURES = [
    # bHLH-LZ family — MYC/MAX related
    #('1NKP', 'A', 'B'),   # MYC-MAX-DNA  ← your primary target
    ('1R05', 'A', 'B'),   # MAD1-MAX bHLH-LZ
    ('4LMY', 'A', 'B'),   # MXD1-MAX
    ('2HNY', 'A', 'B'),   # MAX-DNA (homodimer)
    ('1HLO', 'A', 'B'),   # USF1 bHLH-LZ
    # General protein–protein interfaces (prevents fold overfitting)
    ('1AHW', 'A', 'C'),   # Antibody Fab-protein
    ('2PCC', 'A', 'B'),   # Cytochrome c peroxidase
    ('1EWY', 'A', 'B'),   # IL-4 / receptor
    ('3HHR', 'A', 'B'),   # Growth hormone receptor
    ('1KTZ', 'A', 'B'),   # Thrombin-hirudin
]

RAW_DIR   = '/content/pdb_raw'
CLEAN_DIR = '/content/pdb_clean'
os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

def download_pdb(pdb_id: str) -> str:
    url  = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    path = f'{RAW_DIR}/{pdb_id}.pdb'
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        print(f'  Downloaded {pdb_id}')
    return path

def clean_pdb(pdb_id: str, chain_a: str, chain_b: str) -> str:
    """Strip waters + heteroatoms, keep only the two interface chains."""
    parser  = PDB.PDBParser(QUIET=True)
    struct  = parser.get_structure(pdb_id, f'{RAW_DIR}/{pdb_id}.pdb')
    io      = PDB.PDBIO()

    class ChainSelector(PDB.Select):
        def accept_chain(self, chain):
            return chain.id in (chain_a, chain_b)
        def accept_residue(self, residue):
            return residue.id[0] == ' '   # exclude HETATM and water

    out_path = f'{CLEAN_DIR}/{pdb_id}_clean.pdb'
    io.set_structure(struct)
    io.save(out_path, ChainSelector())
    return out_path

for pdb_id, ca, cb in TRAINING_STRUCTURES:
    download_pdb(pdb_id)
    clean_pdb(pdb_id, ca, cb)

print('\nAll structures downloaded and cleaned.')

  Downloaded 1R05
  Downloaded 4LMY
  Downloaded 2HNY
  Downloaded 1HLO
  Downloaded 1AHW
  Downloaded 2PCC
  Downloaded 1EWY
  Downloaded 3HHR
  Downloaded 1KTZ

All structures downloaded and cleaned.


In [ ]:
# ── Cell 4: Generate SES mesh per chain ───────────────────────────────────────

def generate_ses_mesh(clean_pdb_path: str, chain_id: str, probe_radius: float = 1.4):
    """
    Uses FreeSASA to compute per-atom SASA, then generates a point cloud
    on the SES by placing probe-contact points around each heavy atom.
    Returns:
        verts  [N, 3]  float32 — surface vertex coordinates (Angstroms)
        normals[N, 3]  float32 — outward-pointing unit normals
    """
    parser = PDB.PDBParser(QUIET=True)
    struct = parser.get_structure('X', clean_pdb_path)

    # Collect heavy atoms from the requested chain
    atoms, coords, radii = [], [], []
    vdw = {'C':1.7,'N':1.55,'O':1.52,'S':1.8,'P':1.8,'H':1.2}
    for atom in struct[0][chain_id].get_atoms():
        elem = atom.element.strip() if atom.element else atom.name[0]
        atoms.append(atom)
        coords.append(atom.get_vector().get_array())
        radii.append(vdw.get(elem, 1.7))

    coords  = np.array(coords)   # [M, 3]
    radii   = np.array(radii)

    # Build SES as the boundary of the union of atom spheres + probe
    # We sample points densely on each atom+probe sphere and keep only
    # those that are NOT inside any other atom+probe sphere (Connolly SES).
    extended = radii + probe_radius          # extended radii for sampling
    kdtree   = cKDTree(coords)
    all_verts, all_normals = [], []

    rng  = np.random.default_rng(42)
    N_pts_per_atom = 300

    for i, (c, r) in enumerate(zip(coords, extended)):
        # Sample points uniformly on sphere surface
        theta = np.arccos(1 - 2*rng.random(N_pts_per_atom))
        phi   = 2*np.pi*rng.random(N_pts_per_atom)
        pts   = c + r * np.column_stack([
            np.sin(theta)*np.cos(phi),
            np.sin(theta)*np.sin(phi),
            np.cos(theta)
        ])

        # Keep only points outside all other atom+probe spheres
        dists, idxs = kdtree.query(pts, k=2)  # k=2 because closest is self
        second_dists = dists[:, 1]
        second_idx   = idxs[:, 1]
        mask = second_dists >= extended[second_idx]  # not buried
        if mask.sum() == 0:
            continue

        kept    = pts[mask]
        normals = (kept - c) / np.linalg.norm(kept - c, axis=1, keepdims=True)
        all_verts.append(kept)
        all_normals.append(normals)

    verts   = np.concatenate(all_verts,   axis=0).astype(np.float32)
    normals = np.concatenate(all_normals, axis=0).astype(np.float32)

    # Deduplicate nearby points (merge within 0.5 Å)
    tree = cKDTree(verts)
    used = np.zeros(len(verts), dtype=bool)
    keep = []
    for j in range(len(verts)):
        if not used[j]:
            keep.append(j)
            neighbors = tree.query_ball_point(verts[j], r=0.5)
            for n in neighbors:
                used[n] = True

    return verts[keep], normals[keep]

In [ ]:
# ── Cell 5: Compute shape index from principal curvatures ─────────────────────

def compute_shape_index(verts: np.ndarray, k_neighbors: int = 15) -> np.ndarray:
    """
    For each surface vertex, estimate the shape index using local PCA
    to fit a quadric patch and extract principal curvatures.

    Shape index ∈ [-1, 1]:
      -1 = spherical concave (bowl)    +1 = spherical convex (cap)
       0 = saddle point
    """
    tree = cKDTree(verts)
    shape_indices = np.zeros(len(verts), dtype=np.float32)

    for i, v in enumerate(verts):
        _, idx = tree.query(v, k=k_neighbors+1)
        idx    = idx[1:]              # exclude self
        nbrs   = verts[idx] - v      # center neighborhood

        # PCA → local coordinate frame
        cov   = nbrs.T @ nbrs / k_neighbors
        vals, vecs = np.linalg.eigh(cov)
        # Smallest eigenvector ≈ surface normal
        n     = vecs[:, 0]
        u, w  = vecs[:, 1], vecs[:, 2]

        # Project neighbors into tangent plane and fit z = ax² + bxy + cy²
        xy = nbrs @ np.column_stack([u, w])   # [k, 2]
        z  = nbrs @ n                          # [k]

        # Least-squares fit of quadric coefficients [a, b, c]
        A  = np.column_stack([xy[:,0]**2, xy[:,0]*xy[:,1], xy[:,1]**2])
        try:
            coeffs, _, _, _ = np.linalg.lstsq(A, z, rcond=None)
        except:
            continue
        a, b, c = coeffs

        # Principal curvatures from shape operator eigenvalues
        # H = a + c   (mean curvature, up to sign)
        # K = 4ac - b² (Gaussian curvature)
        H = a + c
        K = 4*a*c - b**2
        disc = max(H**2 - K, 0.0)
        k1   =  H + np.sqrt(disc)
        k2   =  H - np.sqrt(disc)

        if abs(k1 - k2) > 1e-6:
            si = (2.0/np.pi) * np.arctan((k1 + k2) / (k1 - k2))
        else:
            si = 0.0   # umbilical point

        shape_indices[i] = float(np.clip(si, -1.0, 1.0))

    return shape_indices

In [ ]:
# ── Cell 6: Chemical feature mapping ──────────────────────────────────────────

# Kyte-Doolittle hydrophobicity scale (normalized to [-1, 1])
KD_RAW = {
    'ILE':4.5,'VAL':4.2,'LEU':3.8,'PHE':2.8,'CYS':2.5,'MET':1.9,
    'ALA':1.8,'GLY':-0.4,'THR':-0.7,'SER':-0.8,'TRP':-0.9,'TYR':-1.3,
    'PRO':-1.6,'HIS':-3.2,'GLU':-3.5,'GLN':-3.5,'ASP':-3.5,'ASN':-3.5,
    'LYS':-3.9,'ARG':-4.5
}
KD_MIN, KD_MAX = -4.5, 4.5
KD_SCALE = {k: (v - KD_MIN)/(KD_MAX - KD_MIN)*2 - 1
            for k, v in KD_RAW.items()}

# Partial charges (pdb2pqr AMBER ff) for electrostatics proxy
# Positive = donor heavy atoms, negative = acceptor heavy atoms
CHARGE_ATOMS = {
    # donors: N-H, O-H environments
    'N': +0.41, 'NZ': +0.34, 'NH1': +0.46, 'NH2': +0.46,
    'ND1':+0.36, 'NE2':+0.31, 'NE': +0.34, 'OG': -0.66,
    'OG1':-0.67, 'OH': -0.61,
    # acceptors
    'O': -0.57, 'OD1':-0.62,'OD2':-0.62,'OE1':-0.60,'OE2':-0.60,
    'OXT':-0.80, 'SD': -0.09,
}

HBOND_DONORS    = {'N','NZ','NH1','NH2','ND1','NE2','NE','OG','OG1','OH'}
HBOND_ACCEPTORS = {'O','OD1','OD2','OE1','OE2','OXT','SD','ND1','NE2','OH'}

def map_chemical_features(surface_verts: np.ndarray,
                          clean_pdb_path: str,
                          chain_id: str,
                          cutoff: float = 3.5):
    """
    For each surface vertex, find the nearest heavy atom within `cutoff` Å
    and assign chemical features.

    Returns:
        hydro   [N]  float32   normalized KD hydrophobicity ∈ [-1, 1]
        phi_elec[N]  float32   partial charge proxy ∈ [-1, 1] (approx electrostatics)
        hbd     [N]  float32   H-bond donor probability ∈ [0, 1]
        hba     [N]  float32   H-bond acceptor probability ∈ [0, 1]
    """
    parser = PDB.PDBParser(QUIET=True)
    struct = parser.get_structure('X', clean_pdb_path)

    atom_coords, atom_resnames, atom_names = [], [], []
    for atom in struct[0][chain_id].get_atoms():
        if atom.element == 'H':
            continue
        atom_coords.append(atom.get_vector().get_array())
        atom_resnames.append(atom.get_parent().get_resname())
        atom_names.append(atom.get_name().strip())

    atom_coords = np.array(atom_coords, dtype=np.float32)
    tree        = cKDTree(atom_coords)

    N = len(surface_verts)
    hydro    = np.zeros(N, dtype=np.float32)
    phi_elec = np.zeros(N, dtype=np.float32)
    hbd      = np.zeros(N, dtype=np.float32)
    hba      = np.zeros(N, dtype=np.float32)

    dists, idxs = tree.query(surface_verts, k=1)

    for i, (dist, idx) in enumerate(zip(dists, idxs)):
        if dist > cutoff:
            continue   # vertex too far from any atom → leave as zeros
        resname   = atom_resnames[idx]
        atom_name = atom_names[idx]

        hydro[i]    = KD_SCALE.get(resname, 0.0)
        phi_elec[i] = CHARGE_ATOMS.get(atom_name, 0.0)
        # Scale charge to [-1, 1] (charges span ~ [-0.8, +0.46] in AMBER)
        phi_elec[i] = float(np.clip(phi_elec[i] / 0.8, -1.0, 1.0))

        # Distance-weighted H-bond probability
        weight       = 1.0 - (dist / cutoff)
        hbd[i]       = weight if atom_name in HBOND_DONORS    else 0.0
        hba[i]       = weight if atom_name in HBOND_ACCEPTORS else 0.0

    return hydro, phi_elec, hbd, hba

In [ ]:
# ── Cell 7: Full node feature extraction ──────────────────────────────────────

def extract_node_features(clean_pdb_path: str, chain_id: str):
    """
    Returns:
        verts    [N, 3]  float32  — surface coordinates
        features [N, 5]  float32  — [shape_index, phi_elec, hydro, hbd, hba]
    """
    print(f'  Generating SES mesh for chain {chain_id}...')
    verts, normals = generate_ses_mesh(clean_pdb_path, chain_id)

    print(f'  {len(verts)} surface vertices. Computing shape index...')
    shape_idx = compute_shape_index(verts, k_neighbors=15)

    print(f'  Mapping chemical features...')
    hydro, phi_elec, hbd, hba = map_chemical_features(verts, clean_pdb_path, chain_id)

    features = np.stack([shape_idx, phi_elec, hydro, hbd, hba], axis=1)
    return verts.astype(np.float32), features.astype(np.float32)

In [ ]:
# ── Cell 8: Assign positive / negative patch labels ───────────────────────────
def knn_graph_manual(pos: torch.Tensor, k: int = 9) -> torch.Tensor:
    """
    Pure PyTorch kNN graph. No torch-cluster required.
    pos: [N, 3]
    Returns edge_index: [2, k*N]
    """
    # Pairwise squared distances
    dist_mat = torch.cdist(pos, pos)                    # [N, N]
    dist_mat.fill_diagonal_(float('inf'))               # exclude self-loops
    _, nbr_idx = dist_mat.topk(k, dim=1, largest=False) # [N, k] closest
    row = torch.arange(pos.size(0)).unsqueeze(1).expand_as(nbr_idx).reshape(-1)
    col = nbr_idx.reshape(-1)
    return torch.stack([row, col], dim=0)               # [2, N*k]
def get_interface_vertex_mask(surface_verts_A: np.ndarray,
                               clean_pdb_path: str,
                               chain_b: str,
                               interface_cutoff: float = 5.0) -> np.ndarray:
    """
    A surface vertex on chain A is labelled POSITIVE (1) if it lies within
    `interface_cutoff` Å of any heavy atom on chain B.

    Returns: mask [N] bool
    """
    parser = PDB.PDBParser(QUIET=True)
    struct = parser.get_structure('X', clean_pdb_path)

    # Collect chain B heavy-atom coordinates
    b_coords = []
    for atom in struct[0][chain_b].get_atoms():
        if atom.element != 'H':
            b_coords.append(atom.get_vector().get_array())
    b_coords = np.array(b_coords, dtype=np.float32)

    tree = cKDTree(b_coords)
    dists, _ = tree.query(surface_verts_A, k=1)
    return dists < interface_cutoff


def extract_patches(verts: np.ndarray,
                    features: np.ndarray,
                    interface_mask: np.ndarray,
                    patch_size: int = 80,
                    neg_sample_ratio: float = 3.0):
    """
    Builds a list of torch_geometric.data.Data objects.
    Each Data = one 80-vertex surface patch centred on a seed vertex.

    Positive seeds:  interface vertices (label=1)
    Negative seeds:  random non-interface vertices (label=0),
                     sampled at neg_sample_ratio × n_positive

    Returns: List[Data]
    """
    tree     = cKDTree(verts)
    pos_idx  = np.where(interface_mask)[0]
    neg_pool = np.where(~interface_mask)[0]

    n_neg    = int(len(pos_idx) * neg_sample_ratio)
    neg_idx  = np.random.choice(neg_pool, size=min(n_neg, len(neg_pool)), replace=False)
    seeds    = [(i, 1) for i in pos_idx] + [(i, 0) for i in neg_idx]

    patches  = []
    for seed, label in seeds:
        _, nbr_idx = tree.query(verts[seed], k=patch_size)
        patch_pos  = torch.tensor(verts[nbr_idx],    dtype=torch.float32)
        patch_feat = torch.tensor(features[nbr_idx], dtype=torch.float32)

        # Build kNN graph within patch
        edge_index = knn_graph_manual(patch_pos, k=9)
        row, col   = edge_index
        dist       = (patch_pos[row] - patch_pos[col]).norm(dim=1)
        mask       = dist < 9.0
        edge_index = to_undirected(edge_index[:, mask])

        patches.append(Data(
            x          = patch_feat,
            pos        = patch_pos,
            edge_index = edge_index,
            y          = torch.tensor([label], dtype=torch.float32)
        ))

    return patches

In [ ]:
# ── Cell 9: Build dataset from all training structures ─────────────────────────
import torch
import os
import numpy as np
from tqdm import tqdm

PATCH_DIR = '/content/drive/MyDrive/myc_gnn/patches'
os.makedirs(PATCH_DIR, exist_ok=True)

patch_count  = 0
patch_labels = []

for pdb_id, chain_a, chain_b in tqdm(TRAINING_STRUCTURES, desc='Processing PDBs'):
    clean_path = f'{CLEAN_DIR}/{pdb_id}_clean.pdb'
    try:
        verts, feats = extract_node_features(clean_path, chain_a)
        iface_mask   = get_interface_vertex_mask(verts, clean_path, chain_b)
        patches      = extract_patches(verts, feats, iface_mask,
                                       patch_size=80, neg_sample_ratio=3.0)

        for p in patches:
            torch.save(p, f'{PATCH_DIR}/patch_{patch_count}.pt')
            patch_labels.append(int(p.y.item()))
            patch_count += 1

        n_pos = int(iface_mask.sum())
        print(f'  {pdb_id}: {n_pos} interface verts → {len(patches)} patches saved to disk')

        del patches, verts, feats, iface_mask

    except Exception as e:
        print(f'  SKIPPED {pdb_id}: {e}')

n_pos = sum(patch_labels)
n_neg = len(patch_labels) - n_pos
print(f'\nTotal patches saved: {patch_count}')
print(f'  Positive: {n_pos}')
print(f'  Negative: {n_neg}')

# ── Lazy-loading dataset ───────────────────────────────────────────────────────
from torch_geometric.data import Dataset
from torch_geometric.data import DataLoader

class PatchDataset(Dataset):
    def __init__(self, patch_dir, indices):
        super().__init__()
        self.paths = [f'{patch_dir}/patch_{i}.pt' for i in indices]

    def len(self):
        return len(self.paths)

    def get(self, idx):
        return torch.load(self.paths[idx], weights_only=False)  # PyTorch 2.6 fix

# Shuffle and split indices
all_indices = list(range(patch_count))
np.random.shuffle(all_indices)
split = int(0.8 * len(all_indices))

train_dataset = PatchDataset(PATCH_DIR, all_indices[:split])
val_dataset   = PatchDataset(PATCH_DIR, all_indices[split:])

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Processing PDBs:   0%|          | 0/9 [00:00<?, ?it/s]

  Generating SES mesh for chain A...
  18700 surface vertices. Computing shape index...
  Mapping chemical features...


Processing PDBs:  11%|█         | 1/9 [02:57<23:43, 177.94s/it]

  1R05: 5417 interface verts → 18700 patches saved to disk
  Generating SES mesh for chain A...
  19261 surface vertices. Computing shape index...
  Mapping chemical features...


Processing PDBs:  22%|██▏       | 2/9 [05:49<20:19, 174.17s/it]

...

  1EWY: 2776 interface verts → 11104 patches saved to disk
  Generating SES mesh for chain A...
  20209 surface vertices. Computing shape index...
  Mapping chemical features...


Processing PDBs:  89%|████████▉ | 8/9 [26:32<03:23, 203.07s/it]

  3HHR: 3808 interface verts → 15232 patches saved to disk
  Generating SES mesh for chain A...
  11462 surface vertices. Computing shape index...
  Mapping chemical features...


Processing PDBs: 100%|██████████| 9/9 [28:20<00:00, 188.95s/it]

  1KTZ: 1466 interface verts → 5864 patches saved to disk

Total patches saved: 124181
  Positive: 31881
  Negative: 92300
Train batches: 3105 | Val batches: 777


In [ ]:
# ── Run this BEFORE Cell 10 (model definition) ────────────────────────────────
import shutil, os
from tqdm import tqdm

LOCAL_PATCH_DIR = '/content/patches_local'
os.makedirs(LOCAL_PATCH_DIR, exist_ok=True)

print('Copying patches from Drive to local SSD...')
for i in tqdm(range(patch_count)):
    src = f'{PATCH_DIR}/patch_{i}.pt'
    dst = f'{LOCAL_PATCH_DIR}/patch_{i}.pt'
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

print('Done. Redefining datasets to use local path...')

# Redefine loaders pointing at local path
train_dataset = PatchDataset(LOCAL_PATCH_DIR, all_indices[:split])
val_dataset   = PatchDataset(LOCAL_PATCH_DIR, all_indices[split:])

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,
                           num_workers=4, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                           num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Copying patches from Drive to local SSD...


NameError: name 'patch_count' is not defined

In [ ]:
# ── Cell 10: MaSIF GNN model ───────────────────────────────────────────────────

class MaSIF_GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(5, 32)
        self.bn1   = BatchNorm(32)
        self.conv2 = GCNConv(32, 64)
        self.bn2   = BatchNorm(64)
        self.conv3 = GCNConv(64, 128)
        self.bn3   = BatchNorm(128)

        self.mlp = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return torch.sigmoid(self.mlp(x))

model     = MaSIF_GNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
loss_fn   = nn.BCELoss()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

Model parameters: 19,393


In [ ]:
# ── Cell 11: Training loop ─────────────────────────────────────────────────────

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(device)
            pred  = model(batch.x, batch.edge_index, batch.batch).squeeze()
            loss  = loss_fn(pred, batch.y.squeeze())
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            predicted   = (pred > 0.5).float()
            correct    += (predicted == batch.y.squeeze()).sum().item()
            total      += batch.num_graphs
    return total_loss / total, correct / total


best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, 101):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | '
              f'Train loss {tr_loss:.4f} acc {tr_acc:.3f} | '
              f'Val loss {va_loss:.4f} acc {va_acc:.3f}')

    # Save best checkpoint
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': va_loss,
            'val_acc':  va_acc,
        }, f'{CHECKPOINT_DIR}/masif_gnn_best.pt')

print(f'\nBest val loss: {best_val_loss:.4f}')

Epoch  10 | Train loss 0.4726 acc 0.779 | Val loss 0.4428 acc 0.796
Epoch  20 | Train loss 0.4345 acc 0.799 | Val loss 0.4116 acc 0.813
Epoch  30 | Train loss 0.4002 acc 0.816 | Val loss 0.3551 acc 0.840
Epoch  40 | Train loss 0.3881 acc 0.824 | Val loss 0.3409 acc 0.847
Epoch  50 | Train loss 0.3694 acc 0.833 | Val loss 0.3164 acc 0.862
Epoch  60 | Train loss 0.3592 acc 0.840 | Val loss 0.3165 acc 0.864
Epoch  70 | Train loss 0.3531 acc 0.842 | Val loss 0.3069 acc 0.867


In [ ]:
import urllib.request
from Bio import PDB

# Download
urllib.request.urlretrieve(
    'https://files.rcsb.org/download/1NKP.pdb',
    '/content/pdb_raw/1NKP.pdb'
)

# Clean — same logic as Cell 3
parser = PDB.PDBParser(QUIET=True)
struct = parser.get_structure('1NKP', '/content/pdb_raw/1NKP.pdb')
io     = PDB.PDBIO()

class ChainSelector(PDB.Select):
    def accept_chain(self, chain):
        return chain.id in ('A', 'B')
    def accept_residue(self, residue):
        return residue.id[0] == ' '

io.set_structure(struct)
io.save('/content/pdb_clean/1NKP_clean.pdb', ChainSelector())
print('1NKP downloaded and cleaned.')

1NKP downloaded and cleaned.


In [ ]:
# ── Cell 12: Score the MYC-MAX interface on 1NKP ──────────────────────────────
from scipy.spatial import cKDTree
import torch
import numpy as np

# Load best weights
ckpt = torch.load(
    f'{CHECKPOINT_DIR}/masif_gnn_best.pt',
    map_location=device,
    weights_only=False   # PyTorch 2.6 fix
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
      f"(val loss {ckpt['val_loss']:.4f}, val acc {ckpt['val_acc']:.3f})")

# Extract surface of MYC chain A from 1NKP
# NOTE: 1NKP must be removed from TRAINING_STRUCTURES in Cell 3
# before this inference is scientifically valid — see data leakage note
print('\nExtracting 1NKP surface features...')
verts_1nkp, feats_1nkp = extract_node_features(
    f'{CLEAN_DIR}/1NKP_clean.pdb', 'A'
)

tree    = cKDTree(verts_1nkp)
scores  = np.zeros(len(verts_1nkp), dtype=np.float32)
n_verts = len(verts_1nkp)

print(f'Scoring {n_verts} surface vertices...')

for i in range(n_verts):
    _, nbr_idx = tree.query(verts_1nkp[i], k=80)
    patch_pos  = torch.tensor(verts_1nkp[nbr_idx], dtype=torch.float32)
    patch_feat = torch.tensor(feats_1nkp[nbr_idx], dtype=torch.float32)

    edge_index = knn_graph_manual(patch_pos, k=9)
    row, col   = edge_index
    dist       = (patch_pos[row] - patch_pos[col]).norm(dim=1)
    edge_index = to_undirected(edge_index[:, dist < 9.0])

    data = Data(
        x          = patch_feat,
        edge_index = edge_index,
        batch      = torch.zeros(80, dtype=torch.long)
    ).to(device)

    with torch.no_grad():
        scores[i] = model(data.x, data.edge_index, data.batch).item()

    del data, patch_pos, patch_feat, edge_index, row, col, dist

    if i % 500 == 0:
        torch.cuda.empty_cache()
        print(f'  {i}/{n_verts} vertices scored...', end='\r')

print(f'\nScoring complete.')

# ── Results & export ───────────────────────────────────────────────────────────
threshold      = 0.7
high_conf_mask = scores > threshold
top_verts      = verts_1nkp[high_conf_mask]
top_scores     = scores[high_conf_mask]

print(f'\nHigh-confidence interface vertices (score > {threshold}): {len(top_verts)}')
print(f'  Mean score in hot region:  {top_scores.mean():.3f}')
print(f'  Max score:                 {top_scores.max():.3f}')
print(f'  Coverage of total surface: {100*len(top_verts)/n_verts:.1f}%')

np.save('/content/drive/MyDrive/myc_gnn/myc_interface_verts.npy',   top_verts)
np.save('/content/drive/MyDrive/myc_gnn/myc_surface_scores.npy',    scores)
np.save('/content/drive/MyDrive/myc_gnn/myc_full_surface_verts.npy', verts_1nkp)

print('\nSaved:')
print('  myc_interface_verts.npy   → RFdiffusion hotspot_res input')
print('  myc_surface_scores.npy    → full per-vertex score map for visualization')
print('  myc_full_surface_verts.npy → full surface for ChimeraX rendering')

Loaded checkpoint from epoch 96 (val loss 0.2874, val acc 0.877)

Extracting 1NKP surface features...
  Generating SES mesh for chain A...
  15595 surface vertices. Computing shape index...
  Mapping chemical features...
Scoring 15595 surface vertices...

Scoring complete.

High-confidence interface vertices (score > 0.7): 579
  Mean score in hot region:  0.806
  Max score:                 0.997
  Coverage of total surface: 3.7%

Saved:
  myc_interface_verts.npy   → RFdiffusion hotspot_res input
  myc_surface_scores.npy    → full per-vertex score map for visualization
  myc_full_surface_verts.npy → full surface for ChimeraX rendering


In [ ]:
import numpy as np
from Bio import PDB
from scipy.spatial import cKDTree

scores  = np.load('/content/drive/MyDrive/myc_gnn/myc_surface_scores.npy')
verts   = np.load('/content/drive/MyDrive/myc_gnn/myc_full_surface_verts.npy')

# Get Cα coordinates for every residue in MYC chain A
parser   = PDB.PDBParser(QUIET=True)
struct   = parser.get_structure('1NKP', '/content/pdb_clean/1NKP_clean.pdb')
residues = [r for r in struct[0]['A'] if r.id[0] == ' ' and 'CA' in r]

ca_coords  = np.array([r['CA'].get_vector().get_array() for r in residues])
res_ids    = [r.id[1] for r in residues]   # residue sequence numbers

# For each vertex, assign its score to the nearest Cα
tree           = cKDTree(ca_coords)
_, nearest_res = tree.query(verts, k=1)

# Aggregate: max score per residue (max captures whether ANY nearby vertex fired)
res_scores = np.zeros(len(residues))
for v_idx, r_idx in enumerate(nearest_res):
    res_scores[r_idx] = max(res_scores[r_idx], scores[v_idx])

# Print the hot residues
print('Hot residues (score > 0.7):')
for res_id, score in zip(res_ids, res_scores):
    if score > 0.7:
        print(f'  Residue {res_id}: {score:.3f}')

# Save as a dict for RFdiffusion
hot_residues = [res_ids[i] for i, s in enumerate(res_scores) if s > 0.7]
np.save('/content/drive/MyDrive/myc_gnn/myc_hot_residues.npy', hot_residues)
print(f'\n{len(hot_residues)} hot residues saved → myc_hot_residues.npy')

Hot residues (score > 0.7):
  Residue 897: 0.825
  Residue 898: 0.942
  Residue 899: 0.930
  Residue 900: 0.923
  Residue 901: 0.997
  Residue 902: 0.823
  Residue 903: 0.841
  Residue 904: 0.875
  Residue 905: 0.932
  Residue 906: 0.859
  Residue 908: 0.809
  Residue 909: 0.800
  Residue 910: 0.870
  Residue 911: 0.779
  Residue 914: 0.797
  Residue 916: 0.747
  Residue 917: 0.862
  Residue 918: 0.942
  Residue 919: 0.873
  Residue 920: 0.890
  Residue 921: 0.803
  Residue 922: 0.804
  Residue 923: 0.771
  Residue 927: 0.984
  Residue 929: 0.970
  Residue 933: 0.736
  Residue 934: 0.780
  Residue 935: 0.749
  Residue 938: 0.949
  Residue 939: 0.771
  Residue 940: 0.798
  Residue 943: 0.872
  Residue 946: 0.803
  Residue 947: 0.980
  Residue 948: 0.798
  Residue 949: 0.801
  Residue 950: 0.977
  Residue 951: 0.816
  Residue 952: 0.858
  Residue 953: 0.738
  Residue 954: 0.871
  Residue 957: 0.714
  Residue 958: 0.930
  Residue 960: 0.774
  Residue 961: 0.863
  Residue 962: 0.813
  Resi

In [ ]:
import numpy as np

scores = np.load('/content/drive/MyDrive/myc_gnn/myc_surface_scores.npy')
verts  = np.load('/content/drive/MyDrive/myc_gnn/myc_full_surface_verts.npy')

# Only write vertices above threshold, larger spheres
with open('/content/drive/MyDrive/myc_gnn/myc_hotspot_only.bild', 'w') as f:
    for (x, y, z), score in zip(verts, scores):
        if score > 0.7:
            # Scale color intensity by score
            r = float(score)
            b = float(1.0 - score)
            f.write(f'.color {r:.3f} 0.0 {b:.3f}\n')
            f.write(f'.sphere {x:.3f} {y:.3f} {z:.3f} 1.2\n')  # 1.2Å radius

print('Saved myc_hotspot_only.bild')


Saved myc_hotspot_only.bild


In [ ]:
import numpy as np
import os

DRIVE_DIR = '/content/drive/MyDrive/myc_gnn'

scores_dict = {
    897:0.825, 898:0.942, 899:0.930, 900:0.923, 901:0.997,
    902:0.823, 903:0.841, 904:0.875, 905:0.932, 906:0.859,
    908:0.809, 909:0.800, 910:0.870, 911:0.779, 914:0.797,
    916:0.747, 917:0.862, 918:0.942, 919:0.873, 920:0.890,
    921:0.803, 922:0.804, 923:0.771, 927:0.984, 929:0.970,
    933:0.736, 934:0.780, 935:0.749, 938:0.949, 939:0.771,
    940:0.798, 943:0.872, 946:0.803, 947:0.980, 948:0.798,
    949:0.801, 950:0.977, 951:0.816, 952:0.858, 953:0.738,
    954:0.871, 957:0.714, 958:0.930, 960:0.774, 961:0.863,
    962:0.813, 963:0.734, 964:0.736, 965:0.704, 966:0.887,
    967:0.842, 968:0.756, 970:0.795, 971:0.900, 973:0.751,
    974:0.826, 975:0.872, 976:0.949, 977:0.906, 978:0.931,
    979:0.711, 980:0.832, 981:0.750, 983:0.709, 984:0.730
}

fixed_motif = sorted([r for r, s in scores_dict.items() if s >= 0.95])
hotspot_res = sorted([r for r, s in scores_dict.items() if 0.7 <= s < 0.95])

np.save(f'{DRIVE_DIR}/fixed_motif_residues.npy',   fixed_motif)
np.save(f'{DRIVE_DIR}/hotspot_residues_rfdiff.npy', hotspot_res)

# Print RFdiffusion-ready strings
motif_str   = 'A' + ',A'.join(str(r) for r in fixed_motif)
hotspot_str = '[' + ','.join(f'A{r}' for r in fixed_motif + hotspot_res) + ']'

print('Fixed motif residues:', fixed_motif)
print('Hotspot residues:',     hotspot_res)
print(f'\nRFdiffusion motif string:  {motif_str}')
print(f'RFdiffusion hotspot_res:   {hotspot_str}')
print('\nBoth files saved to Drive.')

Fixed motif residues: [901, 927, 929, 947, 950]
Hotspot residues: [897, 898, 899, 900, 902, 903, 904, 905, 906, 908, 909, 910, 911, 914, 916, 917, 918, 919, 920, 921, 922, 923, 933, 934, 935, 938, 939, 940, 943, 946, 948, 949, 951, 952, 953, 954, 957, 958, 960, 961, 962, 963, 964, 965, 966, 967, 968, 970, 971, 973, 974, 975, 976, 977, 978, 979, 980, 981, 983, 984]

RFdiffusion motif string:  A901,A927,A929,A947,A950
RFdiffusion hotspot_res:   [A901,A927,A929,A947,A950,A897,A898,A899,A900,A902,A903,A904,A905,A906,A908,A909,A910,A911,A914,A916,A917,A918,A919,A920,A921,A922,A923,A933,A934,A935,A938,A939,A940,A943,A946,A948,A949,A951,A952,A953,A954,A957,A958,A960,A961,A962,A963,A964,A965,A966,A967,A968,A970,A971,A973,A974,A975,A976,A977,A978,A979,A980,A981,A983,A984]

Both files saved to Drive.


In [ ]:
from Bio import PDB

parser = PDB.PDBParser(QUIET=True)
struct = parser.get_structure('1NKP', '/content/pdb_clean/1NKP_clean.pdb')

print("PDB residue numbers in chain A (MYC):")
residues = [r for r in struct[0]['A'] if r.id[0] == ' ']
for r in residues:
    print(f"  PDB: {r.id[1]}  AA: {r.get_resname()}")

PDB residue numbers in chain A (MYC):
  PDB: 897  AA: GLY
  PDB: 898  AA: HIS
  PDB: 899  AA: MET
  PDB: 900  AA: ASN
  PDB: 901  AA: VAL
  PDB: 902  AA: LYS
  PDB: 903  AA: ARG
  PDB: 904  AA: ARG
  PDB: 905  AA: THR
  PDB: 906  AA: HIS
  PDB: 907  AA: ASN
  PDB: 908  AA: VAL
  PDB: 909  AA: LEU
  PDB: 910  AA: GLU
  PDB: 911  AA: ARG
  PDB: 912  AA: GLN
  PDB: 913  AA: ARG
  PDB: 914  AA: ARG
  PDB: 915  AA: ASN
  PDB: 916  AA: GLU
  PDB: 917  AA: LEU
  PDB: 918  AA: LYS
  PDB: 919  AA: ARG
  PDB: 920  AA: SER
  PDB: 921  AA: PHE
  PDB: 922  AA: PHE
  PDB: 923  AA: ALA
  PDB: 924  AA: LEU
  PDB: 925  AA: ARG
  PDB: 926  AA: ASP
  PDB: 927  AA: GLN
  PDB: 928  AA: ILE
  PDB: 929  AA: PRO
  PDB: 930  AA: GLU
  PDB: 931  AA: LEU
  PDB: 932  AA: GLU
  PDB: 933  AA: ASN
  PDB: 934  AA: ASN
  PDB: 935  AA: GLU
  PDB: 936  AA: LYS
  PDB: 937  AA: ALA
  PDB: 938  AA: PRO
  PDB: 939  AA: LYS
  PDB: 940  AA: VAL
  PDB: 941  AA: VAL
  PDB: 942  AA: ILE
  PDB: 943  AA: LEU
  PDB: 944  AA: LYS
  

In [12]:
scores_dict = {
    897:0.825, 898:0.942, 899:0.930, 900:0.923, 901:0.997,
    902:0.823, 903:0.841, 904:0.875, 905:0.932, 906:0.859,
    908:0.809, 909:0.800, 910:0.870, 911:0.779, 914:0.797,
    916:0.747, 917:0.862, 918:0.942, 919:0.873, 920:0.890,
    921:0.803, 922:0.804, 923:0.771, 927:0.984, 929:0.970,
    933:0.736, 934:0.780, 935:0.749, 938:0.949, 939:0.771,
    940:0.798, 943:0.872, 946:0.803, 947:0.980, 948:0.798,
    949:0.801, 950:0.977, 951:0.816, 952:0.858, 953:0.738,
    954:0.871, 957:0.714, 958:0.930, 960:0.774, 961:0.863,
    962:0.813, 963:0.734, 964:0.736, 965:0.704, 966:0.887,
    967:0.842, 968:0.756, 970:0.795, 971:0.900, 973:0.751,
    974:0.826, 975:0.872, 976:0.949, 977:0.906, 978:0.931,
    979:0.711, 980:0.832, 981:0.750, 983:0.709, 984:0.730
}

literature_hotspots_pdb = [901, 903, 907, 911, 927, 929, 947, 950, 964, 967, 968]

print("Literature hotspot scores:")
for r in literature_hotspots_pdb:
    score = scores_dict.get(r, "NOT IN HOTSPOT LIST")
    print(f"  Residue {r}: {score}")

Literature hotspot scores:
  Residue 901: 0.997
  Residue 903: 0.841
  Residue 907: NOT IN HOTSPOT LIST
  Residue 911: 0.779
  Residue 927: 0.984
  Residue 929: 0.97
  Residue 947: 0.98
  Residue 950: 0.977
  Residue 964: 0.736
  Residue 967: 0.842
  Residue 968: 0.756


In [13]:
import numpy as np

# Add 964 (Leu420, LZ core d-position — confirmed Krylov 1997)
fixed_motif  = [901, 927, 929, 947, 950]
hotspot_res  = sorted(list(set(
    [897,898,899,900,902,903,904,905,906,908,909,910,
     911,914,916,917,918,919,920,921,922,923,933,934,
     935,938,939,940,943,946,948,949,951,952,953,954,
     957,958,960,961,962,963,964,965,966,967,968,970,
     971,973,974,975,976,977,978,979,980,981,983,984]
)))

motif_str   = 'A' + ',A'.join(str(r) for r in fixed_motif)
hotspot_str = '[' + ','.join(f'A{r}' for r in fixed_motif + hotspot_res) + ']'

np.save('/content/drive/MyDrive/myc_gnn/fixed_motif_residues.npy',   fixed_motif)
np.save('/content/drive/MyDrive/myc_gnn/hotspot_residues_rfdiff.npy', hotspot_res)

print(f'Motif string:   {motif_str}')
print(f'Hotspot string: {hotspot_str}')
print('\nCopy both strings — you paste them directly into the RFdiffusion config.')

Motif string:   A901,A927,A929,A947,A950
Hotspot string: [A901,A927,A929,A947,A950,A897,A898,A899,A900,A902,A903,A904,A905,A906,A908,A909,A910,A911,A914,A916,A917,A918,A919,A920,A921,A922,A923,A933,A934,A935,A938,A939,A940,A943,A946,A948,A949,A951,A952,A953,A954,A957,A958,A960,A961,A962,A963,A964,A965,A966,A967,A968,A970,A971,A973,A974,A975,A976,A977,A978,A979,A980,A981,A983,A984]

Copy both strings — you paste them directly into the RFdiffusion config.


In [14]:
import json
import numpy as np

config = {
    # Model performance
    "gnn_val_accuracy": 0.877,
    "gnn_val_loss": 0.2874,
    "gnn_best_epoch": 96,

    # Fixed motif — locked residues for RFdiffusion
    "fixed_motif_pdb": [901, 927, 929, 947, 950],
    "fixed_motif_str": "A901,A927,A929,A947,A950",

    # Hotspot residues — contact targets for RFdiffusion
    "hotspot_residues_pdb": [
        897,898,899,900,902,903,904,905,906,908,909,910,
        911,914,916,917,918,919,920,921,922,923,933,934,
        935,938,939,940,943,946,948,949,951,952,953,954,
        957,958,960,961,962,963,964,965,966,967,968,970,
        971,973,974,975,976,977,978,979,980,981,983,984
    ],
    "hotspot_str": "[A901,A927,A929,A947,A950,A897,A898,A899,A900,A902,A903,A904,A905,A906,A908,A909,A910,A911,A914,A916,A917,A918,A919,A920,A921,A922,A923,A933,A934,A935,A938,A939,A940,A943,A946,A948,A949,A951,A952,A953,A954,A957,A958,A960,A961,A962,A963,A964,A965,A966,A967,A968,A970,A971,A973,A974,A975,A976,A977,A978,A979,A980,A981,A983,A984]",

    # Per-residue GNN scores — full dict
    "residue_scores": {
        897:0.825, 898:0.942, 899:0.930, 900:0.923, 901:0.997,
        902:0.823, 903:0.841, 904:0.875, 905:0.932, 906:0.859,
        908:0.809, 909:0.800, 910:0.870, 911:0.779, 914:0.797,
        916:0.747, 917:0.862, 918:0.942, 919:0.873, 920:0.890,
        921:0.803, 922:0.804, 923:0.771, 927:0.984, 929:0.970,
        933:0.736, 934:0.780, 935:0.749, 938:0.949, 939:0.771,
        940:0.798, 943:0.872, 946:0.803, 947:0.980, 948:0.798,
        949:0.801, 950:0.977, 951:0.816, 952:0.858, 953:0.738,
        954:0.871, 957:0.714, 958:0.930, 960:0.774, 961:0.863,
        962:0.813, 963:0.734, 964:0.736, 965:0.704, 966:0.887,
        967:0.842, 968:0.756, 970:0.795, 971:0.900, 973:0.751,
        974:0.826, 975:0.872, 976:0.949, 977:0.906, 978:0.931,
        979:0.711, 980:0.832, 981:0.750, 983:0.709, 984:0.730
    },

    # Literature validation
    "literature_hotspots_pdb": [901, 903, 907, 911, 927, 929, 947, 950, 964, 967, 968],
    "literature_overlap_pct": 90.9,
    "residue_907_note": "NOT IN HOTSPOT LIST — DNA contact residue, not PPI contact. Expected miss.",

    # PDB info
    "target_pdb": "1NKP",
    "target_chain": "A",
    "uniprot_id": "P01106",
    "pdb_to_uniprot_offset": -544,

    # File paths on Drive
    "drive_dir": "/content/drive/MyDrive/myc_gnn",
    "checkpoint_path": "/content/drive/MyDrive/myc_gnn/checkpoints/masif_gnn_best.pt",
    "interface_verts_path": "/content/drive/MyDrive/myc_gnn/myc_interface_verts.npy",
    "surface_scores_path": "/content/drive/MyDrive/myc_gnn/myc_surface_scores.npy",
    "full_surface_path": "/content/drive/MyDrive/myc_gnn/myc_full_surface_verts.npy",
    "hot_residues_path": "/content/drive/MyDrive/myc_gnn/myc_hot_residues.npy",
    "fixed_motif_path": "/content/drive/MyDrive/myc_gnn/fixed_motif_residues.npy",
    "hotspot_path": "/content/drive/MyDrive/myc_gnn/hotspot_residues_rfdiff.npy",

    # RFdiffusion parameters — pre-filled from GNN output
    "rfdiff_contigs": "A901-950/0 50-100",
    "rfdiff_hotspot_res": "[A901,A927,A929,A947,A950]",
    "rfdiff_num_designs": 100,
    "cpp_tail_length": 15,
    "cpp_preferred_residues": ["ARG", "LEU"],

    # Training data
    "training_pdbs": ["1R05","4LMY","2HNY","1HLO","1AHW","2PCC","1EWY","3HHR","1KTZ"],
    "held_out_pdb": "1NKP",
    "total_patches": 124181,
    "positive_patches": 31881,
    "negative_patches": 92300,
    "class_ratio": 0.345,
}

# Save as JSON
config_path = '/content/drive/MyDrive/myc_gnn/pipeline_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Config saved to {config_path}')
print(f'Keys: {list(config.keys())}')

Config saved to /content/drive/MyDrive/myc_gnn/pipeline_config.json
Keys: ['gnn_val_accuracy', 'gnn_val_loss', 'gnn_best_epoch', 'fixed_motif_pdb', 'fixed_motif_str', 'hotspot_residues_pdb', 'hotspot_str', 'residue_scores', 'literature_hotspots_pdb', 'literature_overlap_pct', 'residue_907_note', 'target_pdb', 'target_chain', 'uniprot_id', 'pdb_to_uniprot_offset', 'drive_dir', 'checkpoint_path', 'interface_verts_path', 'surface_scores_path', 'full_surface_path', 'hot_residues_path', 'fixed_motif_path', 'hotspot_path', 'rfdiff_contigs', 'rfdiff_hotspot_res', 'rfdiff_num_designs', 'cpp_tail_length', 'cpp_preferred_residues', 'training_pdbs', 'held_out_pdb', 'total_patches', 'positive_patches', 'negative_patches', 'class_ratio']
